In [2]:
import tkinter as tk

root = tk.Tk()

# Setting some window properties
root.title("Tk Example")
root.configure(background="yellow")
root.minsize(200, 200)
root.maxsize(500, 500)
root.geometry("300x300+50+50")

root.mainloop()

In [ ]:
# https://stackoverflow.com/questions/41656176/tkinter-canvas-zoom-move-pan

In [5]:
from collections import deque
import math
import tkinter as tk
from tkinter import filedialog
import numpy as np
from PIL import Image, ImageOps, ImageTk
import pdf2image


class MatrixTextEditor:

  def __init__(self, root, file_path):
    self.root = root
    self.root.title("Matrix Text File Editor")
    self.file_path = file_path

    # 1. Load the matrix from the text file using NumPy
    try:
      self.matrix = np.loadtxt(self.file_path, dtype=int)
    except Exception as e:
      print(f"Error loading matrix: {e}")
      self.matrix = np.zeros((100, 100), dtype=int)

    self.height, self.width = self.matrix.shape

    # Normalize matrix values for display if they are 0 and 1
    self.val_max = self.matrix.max()
    if self.val_max <= 1:
      self.display_scale = 255
    else:
      self.display_scale = 1

    self.brush_size = 3
    self.draw_value = 0  # Default to drawing black (erasing artifacts)
    self.raycast_mode = "cross"  # Options: "cross" or "single"

    # Overlay state & caching
    self.overlay_pil = None
    self.overlay_opacity = 0.5
    self.invert_overlay = False
    self.overlay_item = None
    self._cached_overlay_zoom = None
    self._cached_overlay_opacity = None
    self._cached_overlay_invert = None
    self._cached_tk_overlay = None

    # Zoom & Pan State variables
    self.zoom_scale = 1.0
    self.pan_x = 200.0
    self.pan_y = 70.0
    self.drag_start_x = 0
    self.drag_start_y = 0

    # Stroke tracking for interpolation
    self.last_img_x = None
    self.last_img_y = None

    # Cursor preview item reference
    self.cursor_rect = None

    # Caching for performance
    self._cached_zoom = None
    self._cached_tk_img = None

    # Undo / Redo History Stacks
    self.undo_stack = []
    self.redo_stack = []
    self.max_history = 30

    self.setup_ui()
    self.update_canvas()

  def setup_ui(self):
    # Control Panel
    control_frame = tk.Frame(self.root, bg="#f0f0f0", pady=5)
    control_frame.pack(side=tk.TOP, fill=tk.X)

    tk.Button(
        control_frame,
        text="Eraser / Black (0)",
        command=lambda: self.set_draw_value(0),
        bg="#ffcccc",
        font=("Arial", 10, "bold"),
    ).pack(side=tk.LEFT, padx=5)
    tk.Button(
        control_frame,
        text="Draw / White (1/Max)",
        command=lambda: self.set_draw_value(1),
        bg="#ccffcc",
        font=("Arial", 10, "bold"),
    ).pack(side=tk.LEFT, padx=5)

    tk.Label(control_frame, text="Brush Size:", bg="#f0f0f0").pack(
        side=tk.LEFT, padx=10
    )
    self.size_slider = tk.Scale(
        control_frame,
        from_=1,
        to=20,
        orient=tk.HORIZONTAL,
        command=self.update_brush,
    )
    self.size_slider.set(self.brush_size)
    self.size_slider.pack(side=tk.LEFT, padx=5)

    tk.Label(control_frame, text="Raycast:", bg="#f0f0f0").pack(
        side=tk.LEFT, padx=(10, 2)
    )
    self.raycast_mode_var = tk.StringVar(value=self.raycast_mode)
    raycast_menu = tk.OptionMenu(
        control_frame,
        self.raycast_mode_var,
        "cross",
        "single",
        command=self.set_raycast_mode,
    )
    raycast_menu.config(font=("Arial", 9, "bold"), bg="#e6f2ff")
    raycast_menu.pack(side=tk.LEFT, padx=3)

    tk.Button(
        control_frame,
        text="Load Overlay",
        command=self.load_overlay_prompt,
        bg="#fff0b3",
        font=("Arial", 10, "bold"),
    ).pack(side=tk.LEFT, padx=5)

    self.invert_btn = tk.Button(
        control_frame,
        text="Invert Overlay: OFF",
        command=self.toggle_invert_overlay,
        bg="#e6e6fa",
        font=("Arial", 10, "bold"),
    )
    self.invert_btn.pack(side=tk.LEFT, padx=5)

    # tk.Button(
    #     control_frame,
    #     text="Reset View",
    #     command=self.reset_view,
    #     bg="#e0e0e0",
    #     font=("Arial", 10, "bold"),
    # ).pack(side=tk.LEFT, padx=5)

    tk.Button(
        control_frame,
        text="Save Matrix to .txt",
        command=self.save_matrix,
        bg="#cce5ff",
        font=("Arial", 10, "bold"),
    ).pack(side=tk.RIGHT, padx=5)

    # Main Workspace: Vertical Opacity Scrolltab on Left, Canvas on Right
    workspace_frame = tk.Frame(self.root)
    workspace_frame.pack(fill=tk.BOTH, expand=True)

    opacity_frame = tk.Frame(workspace_frame, bg="#f0f0f0", padx=8, pady=8)
    opacity_frame.pack(side=tk.LEFT, fill=tk.Y)

    tk.Label(
        opacity_frame,
        text="Overlay\nOpacity",
        bg="#f0f0f0",
        font=("Arial", 9, "bold"),
        justify=tk.CENTER,
    ).pack(side=tk.TOP, pady=(0, 4))

    self.opacity_value_label = tk.Label(
        opacity_frame,
        text=f"{int(self.overlay_opacity * 100)}%",
        bg="#f0f0f0",
        font=("Arial", 9),
    )
    self.opacity_value_label.pack(side=tk.TOP, pady=(0, 6))

    # Vertical slider stretched vertically and centered cleanly
    self.opacity_slider = tk.Scale(
        opacity_frame,
        from_=100,
        to=0,
        orient=tk.VERTICAL,
        showvalue=False,
        command=self.update_opacity,
    )
    self.opacity_slider.set(int(self.overlay_opacity * 100))
    self.opacity_slider.pack(side=tk.TOP, fill=tk.Y, expand=True)

    self.canvas = tk.Canvas(workspace_frame, bg="gray", highlightthickness=0)
    self.canvas.pack(side=tk.LEFT, fill=tk.BOTH, expand=True)

    # Bind Mouse Actions for Painting (Left Click & Drag)
    self.canvas.bind("<Button-1>", self.start_stroke)
    self.canvas.bind("<B1-Motion>", self.paint)
    self.canvas.bind("<ButtonRelease-1>", self.end_stroke)

    # Bind Right-Click to Ray-Cast Line Inspection
    self.canvas.bind("<Button-3>", self.run_ray_cast_at_click)

    # Bind Middle-Click for Panning
    self.canvas.bind("<Button-2>", self.start_pan)
    self.canvas.bind("<B2-Motion>", self.do_pan)

    # Bind Mouse Wheel for Zooming
    self.canvas.bind("<MouseWheel>", self.zoom)  # Windows and macOS
    self.canvas.bind("<Button-4>", self.zoom)  # Linux scroll up
    self.canvas.bind("<Button-5>", self.zoom)  # Linux scroll down

    # Bind Cursor Hover Preview Events & Hide OS Mouse
    self.canvas.bind("<Motion>", self.update_cursor)
    self.canvas.bind("<Enter>", self.show_cursor)
    self.canvas.bind("<Leave>", self.hide_cursor)

    # Bind Keyboard Shortcuts for Undo (Ctrl+Z / Cmd+Z) and Redo (Ctrl+Y / Ctrl+Shift+Z)
    self.root.bind("<Control-z>", self.undo)
    self.root.bind("<Command-z>", self.undo)  # macOS support
    self.root.bind("<Control-y>", self.redo)
    self.root.bind("<Control-Shift-Z>", self.redo)

  def set_raycast_mode(self, mode):
    self.raycast_mode = mode
    print(f"Raycast mode set to: {self.raycast_mode}")

  def load_overlay_prompt(self):
    file_path = filedialog.askopenfilename(
        title="Select Overlay Image or PDF",
        filetypes=[
            (
                "Supported Files",
                "*.png *.jpg *.jpeg *.bmp *.tif *.tiff *.pdf",
            ),
            ("Image Files", "*.png *.jpg *.jpeg *.bmp *.tif *.tiff"),
            ("PDF Files", "*.pdf"),
            ("All Files", "*.*"),
        ],
    )
    if not file_path:
      return

    try:
      if file_path.lower().endswith(".pdf"):
        pages = pdf2image.convert_from_path(file_path, first_page=1, last_page=1)
        if not pages:
          print("No pages found in PDF.")
          return
        img = pages[0].convert("RGBA")
      else:
        img = Image.open(file_path).convert("RGBA")

      # Resize overlay to match matrix dimensions so it aligns with coordinates
      self.overlay_pil = img.resize(
          (self.width, self.height), Image.Resampling.LANCZOS
      )
      self._cached_overlay_zoom = None
      self._cached_overlay_opacity = None
      self._cached_overlay_invert = None
      self._cached_tk_overlay = None
      self.update_canvas(matrix_changed=False)
      print(f"Loaded overlay: {file_path}")
    except Exception as e:
      print(f"Error loading overlay: {e}")

  def toggle_invert_overlay(self):
    self.invert_overlay = not self.invert_overlay
    state_text = "ON" if self.invert_overlay else "OFF"
    self.invert_btn.config(text=f"Invert Overlay: {state_text}")
    self._cached_tk_overlay = None
    if self.overlay_pil is not None:
      self.update_canvas(matrix_changed=False)

  def update_opacity(self, val):
    pct = int(val)
    self.overlay_opacity = pct / 100.0
    if hasattr(self, "opacity_value_label"):
      self.opacity_value_label.config(text=f"{pct}%")
    if self.overlay_pil is not None:
      self.update_canvas(matrix_changed=False)

  def _get_overlay_photoimage(self, new_width, new_height):
    if self.overlay_pil is None or self.overlay_opacity <= 0.0:
      return None

    if (
        self._cached_tk_overlay is None
        or self._cached_overlay_zoom != (new_width, new_height)
        or self._cached_overlay_opacity != self.overlay_opacity
        or self._cached_overlay_invert != self.invert_overlay
    ):
      resized = self.overlay_pil.resize(
          (new_width, new_height), Image.Resampling.NEAREST
      )
      rgba = resized.copy()
      alpha = rgba.split()[3]

      if self.invert_overlay:
        inverted_rgb = ImageOps.invert(rgba.convert("RGB"))
        rgba = Image.merge("RGBA", (*inverted_rgb.split(), alpha))

      scaled_alpha = alpha.point(lambda p: int(p * self.overlay_opacity))
      rgba.putalpha(scaled_alpha)

      self._cached_tk_overlay = ImageTk.PhotoImage(rgba)
      self._cached_overlay_zoom = (new_width, new_height)
      self._cached_overlay_opacity = self.overlay_opacity
      self._cached_overlay_invert = self.invert_overlay

    return self._cached_tk_overlay

  def set_draw_value(self, val):
    if val == 1 and self.val_max > 1:
      self.draw_value = self.val_max
    else:
      self.draw_value = val

  def update_brush(self, val):
    self.brush_size = int(val)

  # def reset_view(self):
  #   self.zoom_scale = 1.0
  #   self.pan_x = 0.0
  #   self.pan_y = 0.0
  #   self.update_canvas()

  def push_undo_state(self):
    self.undo_stack.append(self.matrix.copy())
    if len(self.undo_stack) > self.max_history:
      self.undo_stack.pop(0)
    self.redo_stack.clear()

  def undo(self, event=None):
    if not self.undo_stack:
      return
    self.redo_stack.append(self.matrix.copy())
    self.matrix = self.undo_stack.pop()
    self.update_canvas(matrix_changed=True)

  def redo(self, event=None):
    if not self.redo_stack:
      return
    self.undo_stack.append(self.matrix.copy())
    self.matrix = self.redo_stack.pop()
    self.update_canvas(matrix_changed=True)

  def update_canvas(self, matrix_changed=False):
    scaled_matrix = (self.matrix * self.display_scale).astype(np.uint8)
    pil_img = Image.fromarray(scaled_matrix)

    new_width = max(1, int(self.width * self.zoom_scale))
    new_height = max(1, int(self.height * self.zoom_scale))

    if (
        matrix_changed
        or self._cached_zoom != self.zoom_scale
        or self._cached_tk_img is None
    ):
      resized_pil = pil_img.resize((new_width, new_height), Image.NEAREST)
      self._cached_tk_img = ImageTk.PhotoImage(resized_pil)
      self._cached_zoom = self.zoom_scale

    self.canvas.delete("all")
    self.image_item = self.canvas.create_image(
        self.pan_x, self.pan_y, anchor=tk.NW, image=self._cached_tk_img
    )

    tk_overlay = self._get_overlay_photoimage(new_width, new_height)
    if tk_overlay is not None:
      self.overlay_item = self.canvas.create_image(
          self.pan_x, self.pan_y, anchor=tk.NW, image=tk_overlay
      )
    else:
      self.overlay_item = None

    if self.cursor_rect is not None:
      self.cursor_rect = self.canvas.create_rectangle(
          0, 0, 0, 0, outline="red", width=1
      )

  def show_cursor(self, event):
    self.canvas.config(cursor="none")
    if self.cursor_rect is None:
      self.cursor_rect = self.canvas.create_rectangle(
          0, 0, 0, 0, outline="red", width=1
      )
    self.update_cursor(event)

  def hide_cursor(self, event):
    self.canvas.config(cursor="")
    if self.cursor_rect is not None:
      self.canvas.delete(self.cursor_rect)
      self.cursor_rect = None

  def update_cursor(self, event):
    if self.cursor_rect is None:
      self.show_cursor(event)

    img_x = (event.x - self.pan_x) / self.zoom_scale
    img_y = (event.y - self.pan_y) / self.zoom_scale

    x, y = int(img_x), int(img_y)
    r = self.brush_size

    x1 = (x - r) * self.zoom_scale + self.pan_x
    y1 = (y - r) * self.zoom_scale + self.pan_y
    x2 = (x + r) * self.zoom_scale + self.pan_x
    y2 = (y + r) * self.zoom_scale + self.pan_y

    self.canvas.coords(self.cursor_rect, x1, y1, x2, y2)

  def start_pan(self, event):
    self.drag_start_x = event.x
    self.drag_start_y = event.y
    # Clear temporary raycast overlays when panning starts
    self.update_canvas(matrix_changed=True)

  def do_pan(self, event):
    dx = event.x - self.drag_start_x
    dy = event.y - self.drag_start_y
    self.pan_x += dx
    self.pan_y += dy
    self.drag_start_x = event.x
    self.drag_start_y = event.y

    self.canvas.coords(self.image_item, self.pan_x, self.pan_y)
    if self.overlay_item is not None:
      self.canvas.coords(self.overlay_item, self.pan_x, self.pan_y)
    self.update_cursor(event)

  def zoom(self, event):
    if event.num == 4 or event.delta > 0:
      zoom_factor = 1.15
    elif event.num == 5 or event.delta < 0:
      zoom_factor = 1 / 1.15
    else:
      zoom_factor = 1.0

    old_zoom = self.zoom_scale
    new_zoom = self.zoom_scale * zoom_factor

    if new_zoom < 0.1 or new_zoom > 50.0:
      return

    self.zoom_scale = new_zoom
    self.pan_x = event.x - (event.x - self.pan_x) * (
        self.zoom_scale / old_zoom
    )
    self.pan_y = event.y - (event.y - self.pan_y) * (
        self.zoom_scale / old_zoom
    )

    self.update_canvas(matrix_changed=False)
    self.update_cursor(event)

  def start_stroke(self, event):
    self.push_undo_state()

    img_x = (event.x - self.pan_x) / self.zoom_scale
    img_y = (event.y - self.pan_y) / self.zoom_scale
    self.last_img_x = img_x
    self.last_img_y = img_y
    self.paint(event)

  def paint(self, event):
    img_x = (event.x - self.pan_x) / self.zoom_scale
    img_y = (event.y - self.pan_y) / self.zoom_scale

    if self.last_img_x is None or self.last_img_y is None:
      self.last_img_x = img_x
      self.last_img_y = img_y

    dist = np.hypot(img_x - self.last_img_x, img_y - self.last_img_y)
    steps = max(1, int(dist))

    xs = np.linspace(self.last_img_x, img_x, steps)
    ys = np.linspace(self.last_img_y, img_y, steps)

    matrix_val = (
        self.draw_value
        if self.display_scale == 1
        else (1 if self.draw_value > 0 else 0)
    )
    r = self.brush_size

    for cx, cy in zip(xs, ys):
      x, y = int(cx), int(cy)
      y1, y2 = max(0, y - r), min(self.height, y + r)
      x1, x2 = max(0, x - r), min(self.width, x + r)
      self.matrix[y1:y2, x1:x2] = matrix_val

    self.last_img_x = img_x
    self.last_img_y = img_y

    self.update_canvas(matrix_changed=True)
    self.update_cursor(event)

  def end_stroke(self, event):
    self.last_img_x = None
    self.last_img_y = None

  def find_routes_and_holes(self, seed_point, num_rays=360):
    """Casts rays outward from a given seed point (y, x)."""
    height, width = self.matrix.shape
    cy, cx = seed_point
    ray_results = []

    max_distance = int(math.hypot(height, width))

    for angle in range(num_rays):
      rad = math.radians(angle * (360.0 / num_rays))
      end_y, end_x = cy, cx
      escaped = True

      for r in range(1, max_distance):
        ny = int(cy + r * math.sin(rad))
        nx = int(cx + r * math.cos(rad))

        if not (0 <= ny < height and 0 <= nx < width):
          end_y, end_x = ny, nx
          escaped = True
          break

        if self.matrix[ny, nx] == 1:
          end_y, end_x = ny, nx
          escaped = False
          break

      ray_results.append({
          "start": (cy, cx),
          "end": (end_y, end_x),
          "escaped": escaped,
      })

    return ray_results

  def cast_cardinal_rays(self, center_point):
    """Casts rays strictly in 4 cardinal directions and returns the endpoints

    along with all intermediate points along the cross lines.
    """
    height, width = self.matrix.shape
    cy, cx = center_point
    cross_endpoints = []
    cross_points = [(cy, cx)]
    directions = [(-1, 0), (1, 0), (0, -1), (0, 1)]  # Up, Down, Left, Right
    max_distance = max(height, width)

    for dy, dx in directions:
      end_y, end_x = cy, cx
      for r in range(1, max_distance):
        ny = cy + r * dy
        nx = cx + r * dx
        cross_points.append((ny, nx))
        if not (0 <= ny < height and 0 <= nx < width):
          end_y, end_x = ny, nx
          break
        if self.matrix[ny, nx] == 1:
          end_y, end_x = ny, nx
          break
      cross_endpoints.append((end_y, end_x))

    return cross_endpoints, cross_points

  def run_ray_cast_at_click(self, event):
    """Triggered by Right-Click: Runs single-point or cross-based raycasting."""
    img_x = (event.x - self.pan_x) / self.zoom_scale
    img_y = (event.y - self.pan_y) / self.zoom_scale

    x, y = int(img_x), int(img_y)

    if not (0 <= x < self.width and 0 <= y < self.height):
      print("Clicked outside matrix boundaries!")
      return

    if self.matrix[y, x] == 1:
      print(
          f"Clicked coordinate ({y}, {x}) is inside a wall! Click on open"
          " space (room)."
      )
      return

    # Refresh canvas to clear previous drawings
    self.update_canvas(matrix_changed=True)

    seed_screen_x = x * self.zoom_scale + self.pan_x
    seed_screen_y = y * self.zoom_scale + self.pan_y

    if self.raycast_mode == "single":
      rays = self.find_routes_and_holes((y, x), num_rays=540)
      for ray in rays:
        ey, ex = ray["end"]
        ey = max(0, min(self.height - 1, ey))
        ex = max(0, min(self.width - 1, ex))

        end_screen_x = ex * self.zoom_scale + self.pan_x
        end_screen_y = ey * self.zoom_scale + self.pan_y

        if ray["escaped"]:
          line_color = "yellow"
          line_width = 1.5
        else:
          line_color = "#00cccc"
          line_width = 1.0

        self.canvas.create_line(
            seed_screen_x,
            seed_screen_y,
            end_screen_x,
            end_screen_y,
            fill=line_color,
            width=line_width,
        )
    else:
      # 1. Get the cardinal cross endpoints and all points along the cross lines
      cardinal_ends, cross_points = self.cast_cardinal_rays((y, x))

      # 2. Sample points along the cross to act as additional raycast origins
      sampled_seeds = cross_points[::2]

      # 3. Run ray casting from each seed point on the cross
      all_rays = []
      for seed in sampled_seeds:
        rays = self.find_routes_and_holes(seed, num_rays=180)
        all_rays.extend(rays)

      for ray in all_rays:
        scy, scx = ray["start"]
        ey, ex = ray["end"]

        start_screen_x = scx * self.zoom_scale + self.pan_x
        start_screen_y = scy * self.zoom_scale + self.pan_y
        end_screen_x = ex * self.zoom_scale + self.pan_x
        end_screen_y = ey * self.zoom_scale + self.pan_y

        if ray["escaped"]:
          line_color = "yellow"
          line_width = 1.0
        else:
          line_color = "#00cccc"
          line_width = 0.75

        self.canvas.create_line(
            start_screen_x,
            start_screen_y,
            end_screen_x,
            end_screen_y,
            fill=line_color,
            width=line_width,
        )

      # 4. Draw the prominent Red Cardinal Cross lines on top
      for ey, ex in cardinal_ends:
        ey = max(0, min(self.height - 1, ey))
        ex = max(0, min(self.width - 1, ex))

        end_screen_x = ex * self.zoom_scale + self.pan_x
        end_screen_y = ey * self.zoom_scale + self.pan_y

        self.canvas.create_line(
            seed_screen_x,
            seed_screen_y,
            end_screen_x,
            end_screen_y,
            fill="red",
            width=2.5,
        )

    # 5. Mark the clicked room seed point with a red dot on top
    self.canvas.create_oval(
        seed_screen_x - 4,
        seed_screen_y - 4,
        seed_screen_x + 4,
        seed_screen_y + 4,
        outline="red",
        fill="red",
    )

  def save_matrix(self):
    save_path = filedialog.asksaveasfilename(
        defaultextension=".txt",
        initialfile=self.file_path,
        filetypes=[("Text Files", "*.txt"), ("All Files", "*.*")],
    )
    if save_path:
      np.savetxt(save_path, self.matrix, fmt="%d")
      print(f"Matrix successfully saved to {save_path}")


if __name__ == "__main__":
  root = tk.Tk()
  root.geometry("1280x720")

  target_txt_file = r"../file_io/line_img_matrix.txt"

  app = MatrixTextEditor(root, target_txt_file)
  root.mainloop()

Raycast mode set to: single
Loaded overlay: C:/Users/baoph/Documents/GitHub/IndoorNav_matrix_editor/file_io/page_0.png
